# EDA - Problema 1: Player Role Classification

Notebook de análisis exploratorio y diagnóstico para el dataset de clasificación de roles en Clash of Clans.

El dataset se espera en `data/processed/role_classification.parquet` y contiene una fila por relación jugador-clan.

## 1. Setup & Carga de Datos

Se importan las librerías necesarias y se carga el dataset desde Parquet. Se verifica la integridad básica y los tipos de datos.

In [ ]:
# === 1. Setup & Carga de Datos ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from pathlib import Path
from collections import defaultdict

sns.set_theme(style="whitegrid", palette="viridis")
%matplotlib inline

# Resolver la ruta del dataset desde el directorio de trabajo actual o sus padres
def find_project_root() -> Path:
    """Busca la raíz del proyecto comprobando la existencia del parquet."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for cand in candidates:
        if (cand / 'data' / 'processed' / 'role_classification.parquet').exists():
            return cand
    raise FileNotFoundError(
        "No se encontró data/processed/role_classification.parquet. "
        "Ajusta la ruta o ejecuta el notebook desde la raíz del proyecto."
    )

root = find_project_root()
DATA_PATH = root / 'data' / 'processed' / 'role_classification.parquet'
print(f"Directorio raíz del proyecto: {root}")
print(f"Dataset: {DATA_PATH}")

# Carga del dataset
df = pd.read_parquet(DATA_PATH)
print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

In [ ]:
# Información general de integridad
print("=== Información general ===")
df.info(show_counts=True)

print("\n=== Tipos de datos ===")
display(df.dtypes.to_frame(name='dtype'))

## 2. Auditoría Básica del Dataset

Se verifican dimensiones exactas, tipos, nulos, duplicados y features casi constantes.

In [ ]:
# === 2. Auditoría Básica del Dataset ===
print("Dimensiones exactas:")
print(f"  - Filas: {df.shape[0]}")
print(f"  - Columnas: {df.shape[1]}")

# Valores nulos
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'nulos': missing, 'porcentaje': missing_pct})
print("\nValores nulos por columna (solo columnas con nulos):")
display(missing_df[missing_df['nulos'] > 0].sort_values('porcentaje', ascending=False))

# Filas duplicadas exactas
dup_rows = df.duplicated().sum()
print(f"\nFilas duplicadas exactas: {dup_rows}")

# Features casi constantes
threshold = 0.95
constant_features = []
for col in df.columns:
    value_counts = df[col].value_counts(dropna=False)
    top_freq = value_counts.iloc[0] / len(df) if len(value_counts) > 0 else 1.0
    if top_freq >= threshold:
        constant_features.append((col, 'casi_constante', top_freq))
    if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) <= 1:
        constant_features.append((col, 'varianza_cero', top_freq))

if constant_features:
    print("\nFeatures con valor único o casi constante (>=95% un solo valor):")
    display(pd.DataFrame(constant_features, columns=['columna', 'tipo', 'frecuencia_moda']))
else:
    print("\nNo se detectaron features casi constantes que superen el 95% de un valor único.")

## 3. Análisis del Target (`role`)

Se analiza la distribución de la variable objetivo y se diagnostica el desbalanceo de clases.

In [ ]:
TARGET = 'role'

# Conteos absolutos y porcentuales
target_counts = df[TARGET].value_counts(dropna=False)
target_pct = df[TARGET].value_counts(dropna=False, normalize=True) * 100
target_summary = pd.DataFrame({
    'count': target_counts,
    'percentage': target_pct
})
print("Distribución del target:")
display(target_summary)

# Gráfico de barras
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x=TARGET, order=target_counts.index)
plt.title(f'Distribución de la variable objetivo: {TARGET}')
plt.xlabel(TARGET)
plt.ylabel('Frecuencia')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Diagnóstico de desbalanceo
print("\nDiagnóstico de desbalanceo:")
max_pct = target_pct.max()
min_pct = target_pct.min()
if max_pct > 60:
    print("- Desbalanceo alto: la clase mayoritaria supera el 60%.")
    print("- Considerar class_weight='balanced', SMOTE o elegir métricas como F1-macro / PR-AUC en lugar de accuracy.")
elif max_pct > 45:
    print("- Desbalanceo moderado: evaluar impacto en métricas y considerar estratificación robusta.")
else:
    print("- Distribución relativamente equilibrada; se puede monitorear accuracy/macro-F1.")

## 4. Análisis Funcional de Features

Agrupación conceptual de variables, análisis de asimetría y distribución de variables críticas.

In [ ]:
# === 4. Análisis Funcional de Features ===

def conceptual_group(col: str) -> str:
    """Asigna una columna a un grupo conceptual según su nombre."""
    col_l = col.lower()
    if any(k in col_l for k in ['troop', 'hero', 'spell', 'equipment', 'builder_hall', 'town_hall', 'exp_level', 'achievement']):
        return 'Progreso'
    if any(k in col_l for k in ['donat', 'attack', 'defense', 'war_stars', 'versus_battle']):
        return 'Actividad'
    if any(k in col_l for k in ['loot', 'gold', 'elixir', 'dark_elixir', 'clan_games']):
        return 'Economía'
    if any(k in col_l for k in ['troph', 'clan_rank', 'war_win', 'clan_war_league', 'legend']):
        return 'Métricas de Clan/Competición'
    return 'Otras'

group_to_cols = defaultdict(list)
for col in df.columns:
    group_to_cols[conceptual_group(col)].append(col)

print("Agrupación conceptual de features:")
for grp, cols in group_to_cols.items():
    print(f"\n{grp}:")
    for c in cols:
        print(f"  - {c}")

# Variables numéricas excluyendo el target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in numeric_cols:
    numeric_cols.remove(TARGET)

# Asimetría, % ceros y % NaN
skew_df = pd.DataFrame({
    'skewness': df[numeric_cols].skew(),
    'zero_pct': (df[numeric_cols] == 0).mean() * 100,
    'nan_pct': df[numeric_cols].isna().mean() * 100
}).sort_values('skewness', key=lambda s: s.abs(), ascending=False)

print("\nAsimetría y presencia de ceros/NaN en variables numéricas:")
display(skew_df)

# Visualización de distribuciones críticas
critical_vars = ['donations', 'attack_wins', 'trophies']
present_critical = [v for v in critical_vars if v in df.columns]
if present_critical:
    fig, axes = plt.subplots(1, len(present_critical), figsize=(5 * len(present_critical), 4))
    if len(present_critical) == 1:
        axes = [axes]
    for ax, var in zip(axes, present_critical):
        sns.histplot(data=df, x=var, kde=True, ax=ax)
        ax.set_title(f'Distribución de {var}')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron las variables críticas esperadas (donations, attack_wins, trophies).")

## 5. Relación Feature ↔ Target

Se analiza la separación de clases y se estima el poder predictivo no lineal mediante Mutual Information y Kruskal-Wallis.

In [ ]:
# === 5. Relación Feature ↔ Target ===

# Codificar target para Mutual Information
target_encoded = pd.factorize(df[TARGET])[0]

# Preparar datos numéricos imputando temporalmente la mediana sólo para MI
df_num_for_mi = df[numeric_cols].copy()
for col in numeric_cols:
    if df_num_for_mi[col].isna().any():
        df_num_for_mi[col] = df_num_for_mi[col].fillna(df_num_for_mi[col].median())

# Mutual Information
mi_scores = mutual_info_classif(df_num_for_mi, target_encoded, random_state=42)
mi_df = pd.DataFrame({'feature': numeric_cols, 'mutual_info': mi_scores})
mi_df = mi_df.sort_values('mutual_info', ascending=False).reset_index(drop=True)
print("Poder predictivo no lineal (Mutual Information) por feature numérica:")
display(mi_df.head(20))

# Kruskal-Wallis
kw_results = []
for col in numeric_cols:
    groups = []
    for role in df[TARGET].dropna().unique():
        group_vals = df.loc[df[TARGET] == role, col].dropna()
        if len(group_vals) > 5:
            groups.append(group_vals)
    if len(groups) >= 2:
        try:
            stat, p = stats.kruskal(*groups)
            kw_results.append((col, stat, p))
        except Exception:
            pass

kw_df = pd.DataFrame(kw_results, columns=['feature', 'kruskal_stat', 'p_value'])
kw_df['p_bonferroni'] = kw_df['p_value'] * len(numeric_cols)
kw_df = kw_df.sort_values('p_value').reset_index(drop=True)
print("\nPrueba de Kruskal-Wallis (variables numéricas vs role):")
display(kw_df.head(20))

# Boxplots de las features con mayor Mutual Information
top_mi_features = mi_df.head(6)['feature'].tolist()
if top_mi_features:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for ax, var in zip(axes, top_mi_features):
        sns.boxplot(data=df, x=TARGET, y=var, ax=ax)
        ax.set_title(f'{var} por {TARGET}')
        ax.set_xlabel(TARGET)
        ax.set_ylabel(var)
        plt.setp(ax.get_xticklabels(), rotation=45)
    # Ocultar axes vacíos si hay menos de 6 features
    for ax in axes[len(top_mi_features):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron features numéricas para graficar.")

## 6. Multicolinealidad y Redundancia

Se calculan las matrices de correlación de Pearson y Spearman para identificar pares con |r| > 0.85.

In [ ]:
# === 6. Multicolinealidad y Redundancia ===

df_corr = df[numeric_cols].copy()
df_corr = df_corr.replace([np.inf, -np.inf], np.nan)
for col in df_corr.columns:
    if df_corr[col].isna().any():
        df_corr[col] = df_corr[col].fillna(df_corr[col].median())

pearson_corr = df_corr.corr(method='pearson')
spearman_corr = df_corr.corr(method='spearman')

threshold_corr = 0.85

def get_high_corr_pairs(corr_matrix, threshold=threshold_corr):
    """Devuelve pares de variables con correlación por encima del umbral (triángulo superior)."""
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    pairs = []
    for col in upper.columns:
        for idx in upper.index:
            val = upper.loc[idx, col]
            if pd.notna(val) and abs(val) > threshold:
                pairs.append((idx, col, val))
    return pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'corr'])

pearson_pairs = get_high_corr_pairs(pearson_corr)
spearman_pairs = get_high_corr_pairs(spearman_corr)

print("Pares con |Pearson| > 0.85:")
display(pearson_pairs if len(pearson_pairs) > 0 else None)

print("\nPares con |Spearman| > 0.85:")
display(spearman_pairs if len(spearman_pairs) > 0 else None)

# Heatmap de correlación Pearson
plt.figure(figsize=(12, 10))
sns.heatmap(pearson_corr, annot=False, cmap='coolwarm', center=0, square=True)
plt.title('Matriz de correlación Pearson (variables numéricas)')
plt.tight_layout()
plt.show()

## 7. Detección de Outliers y Casos Extremos

Se identifican outliers mediante la regla IQR y se documentan ejemplos extremos.

In [ ]:
# === 7. Detección de Outliers y Casos Extremos ===

outlier_records = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    n_outliers = int(mask.sum())
    outlier_records.append({
        'feature': col,
        'q1': q1,
        'q3': q3,
        'IQR': iqr,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'n_outliers': n_outliers,
        'pct_outliers': 100.0 * n_outliers / len(df)
    })

outlier_df = pd.DataFrame(outlier_records).sort_values('n_outliers', ascending=False)
print("Resumen de outliers por IQR:")
display(outlier_df)

# Análisis cualitativo para variables críticas
for var in ['donations', 'trophies', 'attack_wins']:
    if var in df.columns:
        top_vals = df.nlargest(10, var)[[var, TARGET]].reset_index(drop=True)
        print(f"\nTop 10 valores extremos de {var}:")
        display(top_vals)

## 8. Modeling Implications (Conclusiones para la Fase de Modelado)

### Resumen estructurado de hallazgos y recomendaciones

In [ ]:
# === Resumen ejecutivo automático ===
summary = {
    'n_rows': df.shape[0],
    'n_cols': df.shape[1],
    'missing_total': int(missing[missing > 0].sum()),
    'constant_features': [c[0] for c in constant_features],
    'target_distribution': target_counts.to_dict(),
    'target_imbalance_ratio': float(max_pct / max(min_pct, 1e-9)),
    'top_mi_features': mi_df.head(10).to_dict('records'),
    'pearson_high_corr_pairs': pearson_pairs.to_dict('records') if len(pearson_pairs) > 0 else [],
    'spearman_high_corr_pairs': spearman_pairs.to_dict('records') if len(spearman_pairs) > 0 else [],
    'outliers_max_pct': float(outlier_df['pct_outliers'].max()) if len(outlier_df) > 0 else 0.0,
    'outliers_most_common': outlier_df.head(1).to_dict('records') if len(outlier_df) > 0 else []
}

print("=== RESUMEN EJECUTIVO EDA P1 ===")
for k, v in summary.items():
    print(f"{k}: {v}")

### Preprocessing Pipeline

- **Imputación**  
  - Variables numéricas: usar la mediana si la asimetría es alta o el porcentaje de outliers es elevado; en caso contrario, la media.  
  - Variables categóricas: usar la moda.  
  - Si más del 40-50% de los valores de una columna son nulos, considerar eliminarla o crear una categoría explícita `"missing"`.

- **Transformaciones**  
  - Aplicar `log1p` a features con asimetría severa (`|skew| > 1`) y presencia de valores muy altos, como `donations`, `trophies` o métricas de botín.  
  - Usar `RobustScaler` para las variables con outliers extremos; `StandardScaler` para el resto.  
  - Codificar el target `role` con `LabelEncoder` (o `OrdinalEncoder` si se prefiere mantener el orden semántico).

### Estrategia de Validación

- Utilizar **`StratifiedKFold`** con 5 o 10 pliegues para preservar la distribución de clases en cada partición.  
- Para conjuntos muy desbalanceados, evitar depender de `accuracy`. Priorizar:  
  - `F1-macro`  
  - `Cohen's Kappa`  
  - `PR-AUC` (para problemas multiclase, usar `average='macro'` o One-vs-Rest).  
  - Matriz de confusión normalizada por filas.

### Feature Pruning

- Eliminar features con **varianza cero** o **casi constantes** (top frecuencia ≥ 95%).  
- Eliminar una feature de cada par con `|Pearson| > 0.85` o `|Spearman| > 0.85`, conservando la que tenga mayor información mutua o interpretabilidad.  
- Revisar concentración de outliers: si `pct_outliers` es muy alto (>10-15%), considerar tratar esa variable con una transformación robusta en lugar de eliminar registros.

### Baseline Recomendado

1. **Regresión Logística multiclase**  
   - `LogisticRegression(multi_class='multinomial', solver='saga', class_weight='balanced', max_iter=1000)`  
   - Buena opción como baseline interpretable y rápida.  

2. **Random Forest**  
   - Robusto ante no linealidades y outliers; usar `class_weight='balanced_subsample'`.  
   - Evaluar importancia de features como complemento al análisis de MI.

3. **Gradient Boosting / XGBoost / LightGBM**  
   - Modelos más potentes si el dataset es grande; usar `scale_pos_weight` o `class_weight` para el desbalanceo y búsqueda de hiperparámetros con validación cruzada anidada.

Estas conclusiones deben alimentar directamente la fase de feature engineering y modelado del Problema 1.